# Question 3 — Building, Benchmarking, and Deploying an Efficient Spelling Corrector

## Objective

The goal of this assignment is to build a spelling corrector that can handle:

1. Non-word spelling errors
2. Real-word spelling errors

The corrector will operate within an edit distance of 1.

Two candidate-generation methods will be implemented:

- **Method A:** Standard Edit Distance 1
- **Method B:** Symmetric Delete Spelling Correction

The two methods will then be compared in terms of:

- Correction accuracy
- Candidate generation
- Execution time

Finally, the spelling corrector will be tested through an interactive terminal-style interface.

## Dataset

The Brown Corpus from NLTK will be used to build:

- Vocabulary
- Unigram frequency model
- Bigram language model

## 1. Setup

First, we import the libraries required for the spelling corrector.

We will use:

- `nltk` for the Brown Corpus
- `collections.Counter` for frequency distributions
- `defaultdict` for storing bigram counts
- `string` for alphabet handling
- `random` for generating corrupted test examples
- `time` for the Speed Demon benchmark

In [1]:
import nltk
import random
import string
import time

from collections import Counter, defaultdict

### Download the Brown Corpus

The assignment requires the Brown Corpus from NLTK.

We download it once and then load its sentences for model construction and evaluation.

In [2]:
nltk.download("brown")

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.


True

In [3]:
from nltk.corpus import brown

sentences = brown.sents()

print("Number of sentences:", len(sentences))
print("First sentence:", sentences[0])

Number of sentences: 57340
First sentence: ['The', 'Fulton', 'County', 'Grand', 'Jury', 'said', 'Friday', 'an', 'investigation', 'of', "Atlanta's", 'recent', 'primary', 'election', 'produced', '``', 'no', 'evidence', "''", 'that', 'any', 'irregularities', 'took', 'place', '.']


## 2. Corpus Preprocessing

The Brown Corpus contains words with punctuation and mixed capitalization.

For this spelling corrector, we will normalize words to lowercase.

Punctuation tokens are excluded from the spelling vocabulary because the vocabulary should primarily represent words that can be corrected.

This preprocessing will be used consistently when building the unigram and bigram models.

In [4]:
def normalize_word(word):
    """
    Normalize a Brown Corpus token for the spelling model.
    """
    return word.lower()


def is_word(word):
    """
    Keep only alphabetic tokens.
    """
    return word.isalpha()


processed_sentences = []

for sentence in sentences:
    words = [
        normalize_word(word)
        for word in sentence
        if is_word(word)
    ]

    if words:
        processed_sentences.append(words)


print("Original Brown sentences:", len(sentences))
print("Processed sentences:", len(processed_sentences))
print("First processed sentence:")
print(processed_sentences[0])

Original Brown sentences: 57340
Processed sentences: 56766
First processed sentence:
['the', 'fulton', 'county', 'grand', 'jury', 'said', 'friday', 'an', 'investigation', 'of', 'recent', 'primary', 'election', 'produced', 'no', 'evidence', 'that', 'any', 'irregularities', 'took', 'place']


## 3. Vocabulary and Unigram Frequency Model

The first model required by the assignment is a unigram model.

A unigram model records how frequently each word appears in the corpus.

For example:

```text
the → very high frequency
house → lower frequency
zebra → lower frequency

In [5]:
word_counts = Counter()

for sentence in processed_sentences:
    word_counts.update(sentence)

vocabulary = set(word_counts.keys())

print("Vocabulary size:", len(vocabulary))
print("Most common words:")
print(word_counts.most_common(20))

Vocabulary size: 40234
Most common words:
[('the', 69971), ('of', 36412), ('and', 28853), ('to', 26158), ('a', 23195), ('in', 21337), ('that', 10594), ('is', 10109), ('was', 9815), ('he', 9548), ('for', 9489), ('it', 8760), ('with', 7289), ('as', 7253), ('his', 6996), ('on', 6741), ('be', 6377), ('at', 5372), ('by', 5306), ('i', 5164)]


### Unigram Probabilities

The unigram probability of a word is:

\[
P(w) = \frac{count(w)}{\sum_{v \in V} count(v)}
\]

where:

- `count(w)` is the frequency of the word
- `V` is the vocabulary

For candidate ranking, raw frequency is sufficient because the denominator is the same for all candidates.

In [6]:
total_words = sum(word_counts.values())

unigram_prob = {
    word: count / total_words
    for word, count in word_counts.items()
}

print("P(the) =", unigram_prob.get("the"))
print("P(house) =", unigram_prob.get("house"))

P(the) = 0.07127417705324146
P(house) = 0.0006020070977757315


## 4. Bigram Language Model

The assignment requires a bigram probability model to handle **real-word errors**
by considering the context of a word in a sentence.

### Maximum Likelihood Estimate (MLE)

The raw bigram probability is:

$$P_{MLE}(w_i \mid w_{i-1}) = \frac{C(w_{i-1},\, w_i)}{C(w_{i-1})}$$

### Problem with MLE: Zero Probabilities

For any bigram $(w_{i-1}, w_i)$ not seen in training data, $C(w_{i-1}, w_i) = 0$,
so $P_{MLE} = 0$, and $\log(0) = -\infty$ — which breaks log-probability arithmetic.

### Fix: Add-k (Laplace) Smoothing

We add a small constant $k$ to every bigram count, and compensate in the denominator
by adding $k \times |V|$ (one $k$ for each vocabulary word):

$$P_k(w_i \mid w_{i-1}) = \frac{C(w_{i-1},\, w_i) + k}{C(w_{i-1}) + k \cdot |V|}$$

This guarantees $P_k > 0$ for **every** bigram, even completely unseen ones.
The smoothing constant $k = 0.01$ keeps frequent bigrams dominant while assigning
a small non-zero floor to unseen pairs.

In [7]:
bigram_counts = defaultdict(Counter)
unigram_context_counts = Counter()

for sentence in processed_sentences:
    previous = "<s>"

    for word in sentence:
        bigram_counts[previous][word] += 1
        unigram_context_counts[previous] += 1
        previous = word

    bigram_counts[previous]["</s>"] += 1
    unigram_context_counts[previous] += 1

# vocabulary_size is used in the add-k smoothing denominator
# +2 accounts for the special <s> and </s> boundary tokens
vocabulary_size = len(vocabulary) + 2

print("Example bigrams after 'the':")
print(bigram_counts["the"].most_common(10))
print(f"Vocabulary size (for smoothing denominator): {vocabulary_size:,}")

Example bigrams after 'the':
[('first', 672), ('same', 629), ('most', 427), ('other', 424), ('new', 408), ('united', 394), ('world', 375), ('state', 275), ('two', 272), ('only', 260)]
Vocabulary size (for smoothing denominator): 40,236


In [8]:
# Smoothing constant k for add-k (Laplace) smoothing
# k=1.0  : full Laplace (often over-smooths for NLP tasks)
# k=0.01 : mild smoothing — keeps high-frequency bigrams dominant
#           while giving small non-zero probability to unseen pairs
BIGRAM_K = 0.01


def bigram_probability(previous, word):
    """
    Returns the add-k smoothed bigram probability P(word | previous).

    Formula:
        P(w | prev) = (C(prev, w) + k) / (C(prev) + k × |V|)

    where:
        C(prev, w) = count of the bigram (prev, w) in the corpus
        C(prev)    = count of prev as a context word (row sum)
        k          = smoothing constant (BIGRAM_K)
        |V|        = vocabulary size (vocabulary_size)

    Why add-k smoothing?
      Without smoothing, P = 0 for any unseen bigram.
      log(0) = -∞, which breaks log-probability arithmetic in
      sentence_bigram_log_probability and real_word_context_score.
      Add-k assigns a small non-zero floor probability to every
      bigram, making the model well-defined over the full vocabulary.
    """
    c_bigram  = bigram_counts[previous][word]
    c_context = unigram_context_counts[previous]

    return (c_bigram + BIGRAM_K) / (c_context + BIGRAM_K * vocabulary_size)


# Sanity checks
p_seen   = bigram_probability("the", "quick")
p_unseen = bigram_probability("the", "zxqvw")    # definitely unseen
print(f"P(quick | the)  = {p_seen:.6f}  (seen bigram)")
print(f"P(zxqvw | the)  = {p_unseen:.8f}  (unseen — should be small but > 0)")
print(f"Unseen floor    = {BIGRAM_K / (unigram_context_counts['the'] + BIGRAM_K * vocabulary_size):.8f}")

P(quick | the)  = 0.000071  (seen bigram)
P(zxqvw | the)  = 0.00000014  (unseen — should be small but > 0)
Unseen floor    = 0.00000014


### Bigram Probability Interpretation

For two consecutive words \(w_{i-1}\) and \(w_i\),\ the bigram model estimates:

\[
P(w_i \mid w_{i-1})
=
\frac{C(w_{i-1}, w_i)}
{C(w_{i-1})}
\]

The model is used during real-word error correction. If a word is already present in the vocabulary but is inappropriate in context, candidate corrections are compared using the probability of their surrounding bigrams.

Sentence boundaries are represented using `<s>` and `</s>`.

In [9]:
def sentence_bigram_log_probability(words):
    """
    Calculate the total log probability of a sentence
    using the add-k smoothed bigram model.

    With add-k smoothing, bigram_probability() always returns
    a value > 0, so log() is always defined. The epsilon hack
    is no longer needed.

    Returns log P(w1, w2, ..., wn) = sum of log P(wi | wi-1).
    """
    import math

    words = [word.lower() for word in words]
    sequence = ["<s>"] + words + ["</s>"]

    log_probability = 0.0

    for previous, current in zip(sequence[:-1], sequence[1:]):
        probability = bigram_probability(previous, current)
        log_probability += math.log(probability)

    return log_probability


# Test
test_phrase = ["i", "would", "like", "to", "see", "the", "world"]
print("Bigram log-probability of test phrase:")
print(f"  {sentence_bigram_log_probability(test_phrase):.4f}")

Bigram log-probability of test phrase:
  -28.7052


In [10]:
test_phrase = ["i", "would", "like", "to", "see", "the", "world"]

print(
    "Bigram log probability:",
    sentence_bigram_log_probability(test_phrase)
)

Bigram log probability: -28.705163725586317


## 5. Method A — Standard Edit Distance 1

The first candidate-generation method generates every possible word that is exactly one edit away from the input word.

The four operations are:

1. Deletion
2. Transposition
3. Replacement
4. Insertion

For example, for:

```text
hello
ello
hllo
helo
hell

### Why Damerau-Levenshtein (DP) 

**Damerau-Levenshtein DP** instead computes the *exact minimum cost* to transform
one string into another using a 2-D DP table:

$$d[i, j] = \min \begin{cases}
d[i-1, j] + 1 & \text{(deletion)} \\
d[i, j-1] + 1 & \text{(insertion)} \\
d[i-1, j-1] + \mathbb{1}[s_1[i] \neq s_2[j]] & \text{(substitution)} \\
d[i-2, j-2] + 1 & \text{(transposition, if applicable)}
\end{cases}$$

Crucially, it correctly handles **transpositions** (e.g., `'teh' → 'the'` = 1 edit),
whereas a naive Levenshtein implementation would score this as 2 edits.
Method A now uses DL DP to compute distances from the typo to every vocabulary word,
returning all words within edit distance 1.

In [ ]:
def damerau_levenshtein(s1: str, s2: str) -> int:
    """
    Computes the Damerau-Levenshtein edit distance between two strings
    using **dynamic programming** (DP).

    The DP table d[i, j] stores the minimum number of single-character
    edits required to transform s1[:i+1] into s2[:j+1].

    Four operations are supported:
      - Deletion      : remove a character from s1
      - Insertion     : add a character into s1
      - Substitution  : replace a character in s1 with another
      - Transposition : swap two adjacent characters in s1

    Time complexity  : O(|s1| * |s2|)
    Space complexity : O(|s1| * |s2|)

    Unlike the Norvig-style enumeration (edits1) which generates all
    possible strings 1-edit away, this function computes the *exact*
    minimum-cost transformation between any two given strings.
    """
    len1, len2 = len(s1), len(s2)

    # Initialise boundary conditions of the DP table
    d = {}
    for i in range(-1, len1 + 1):
        d[(i, -1)] = i + 1   # cost of deleting all of s1[:i+1]
    for j in range(-1, len2 + 1):
        d[(-1, j)] = j + 1   # cost of inserting all of s2[:j+1]

    for i in range(len1):
        for j in range(len2):
            cost = 0 if s1[i] == s2[j] else 1
            d[(i, j)] = min(
                d[(i - 1, j)] + 1,        # deletion
                d[(i, j - 1)] + 1,        # insertion
                d[(i - 1, j - 1)] + cost  # substitution
            )
            # Transposition of two adjacent characters
            if i > 0 and j > 0 and s1[i] == s2[j - 1] and s1[i - 1] == s2[j]:
                d[(i, j)] = min(d[(i, j)], d[(i - 2, j - 2)] + 1)

    return d[(len1 - 1, len2 - 1)]

: 

In [ ]:
# Verify DL DP correctness
print("DL('hav',     'have')     =", damerau_levenshtein('hav',     'have'))     # 1 (insertion)
print("DL('teh',     'the')      =", damerau_levenshtein('teh',     'the'))      # 1 (transposition)
print("DL('sea',     'see')      =", damerau_levenshtein('sea',     'see'))      # 1 (substitution)
print("DL('sentnce', 'sentence') =", damerau_levenshtein('sentnce', 'sentence')) # 2
print("DL('ab',      'ba')       =", damerau_levenshtein('ab',      'ba'))       # 1 (transposition)
print("DL('house',   'house')    =", damerau_levenshtein('house',   'house'))    # 0 (exact match)

### Method A — Standard Edit Distance 1 with DL DP Verification

Method A uses a **3-step pipeline** so that Damerau-Levenshtein DP is the
definitive distance measure while staying fast:

| Step | Operation | Cost |
|------|-----------|------|
| 1 | **Enumerate** all edit-1 strings (delete/transpose/replace/insert) | O(L × \|Σ\|) ≈ 400 strings |
| 2 | **Intersect** with vocabulary (hash set lookup) | O(candidates) ≈ instant |
| 3 | **Verify** each surviving candidate with DL DP | O(L²) × ~5 candidates ≈ fast |

Step 3 is the key upgrade over plain Norvig: it filters out any candidate that
happens to be in the vocabulary but requires more than 1 true DL edit
(e.g. a 2-char word accidentally matching a long word across the hash collision).
It also correctly handles **transpositions** — `'teh' → 'the'` is DL distance 1,
which a naive implementation would score as 2.

In [13]:
def _generate_edit1_strings(word):
    """
    Step 1: Generate all strings reachable from `word` by one edit.
    Produces deletions, transpositions, replacements, and insertions.
    Returns raw strings — most will not be real words.
    Complexity: O(L × |Σ|) ≈ 400 strings for an average word.
    """
    alphabet = "abcdefghijklmnopqrstuvwxyz"
    w = word.lower()
    n = len(w)

    deletes    = [w[:i] + w[i+1:]               for i in range(n)]
    transposes = [w[:i] + w[i+1] + w[i] + w[i+2:] for i in range(n - 1)]
    replaces   = [w[:i] + c + w[i+1:]           for i in range(n) for c in alphabet]
    inserts    = [w[:i] + c + w[i:]             for i in range(n + 1) for c in alphabet]

    return set(deletes + transposes + replaces + inserts)


def method_a_candidates(word):
    """
    Method A — Edit Distance 1 with Damerau-Levenshtein DP Verification.

    3-step pipeline:
      Step 1 — Fast enumeration   : generate all ~400 edit-1 strings.
      Step 2 — Vocab intersection : keep only real vocabulary words (~5-10).
      Step 3 — DL DP verification : confirm exact DL distance ≤ 1 for each
                                    surviving candidate using the DP table.

    Why Step 3?
      Enumeration is fast but direction-agnostic — it generates strings at
      edit-distance 1 FROM the typo. The intersection with vocab gives us
      real words. DL DP is then the authoritative, exact verifier that
      runs on only the tiny surviving set, so its O(L²) cost is negligible.

    This means damerau_levenshtein() is the definitive distance function
    for Method A, with no performance penalty.
    """
    word = word.lower()

    # Step 1: enumerate all edit-1 strings (~400 strings, no vocab check yet)
    raw_candidates = _generate_edit1_strings(word)

    # Step 2: intersect with vocabulary (O(1) hash lookup each)
    vocab_candidates = raw_candidates & vocabulary

    # Step 3: verify each with DL DP — only ~5-10 candidates reach this step
    return {
        c for c in vocab_candidates
        if damerau_levenshtein(word, c) <= 1
    }


# --- Demo ---
candidates_a = method_a_candidates("hav")
print("Method A candidates for 'hav':", candidates_a)
print("DL distances:")
for c in sorted(candidates_a):
    print(f"  DL('hav', '{c}') = {damerau_levenshtein('hav', c)}")

Method A candidates for 'hav': {'han', 'cav', 'hap', 'have', 'haw', 'had', 'hay', 'hal', 'ham', 'hev', 'ha', 'has', 'hat'}
DL distances:
  DL('hav', 'cav') = 1
  DL('hav', 'ha') = 1
  DL('hav', 'had') = 1
  DL('hav', 'hal') = 1
  DL('hav', 'ham') = 1
  DL('hav', 'han') = 1
  DL('hav', 'hap') = 1
  DL('hav', 'has') = 1
  DL('hav', 'hat') = 1
  DL('hav', 'have') = 1
  DL('hav', 'haw') = 1
  DL('hav', 'hay') = 1
  DL('hav', 'hev') = 1


## 6. Method B — Symmetric Delete Spelling Correction

The second method is designed to make candidate generation more efficient.

Instead of generating all replacements, insertions, transpositions, and deletions for every query, we preprocess the vocabulary.

For every vocabulary word, we generate all one-character deletions.

We then create an index:

```text
deleted_variant → original vocabulary words
ello → hello
hllo → hello
helo → hello
hell → hello

In [14]:
delete_index = defaultdict(set)

for word in vocabulary:
    for i in range(len(word)):
        deleted = word[:i] + word[i+1:]
        delete_index[deleted].add(word)

print("Delete index size:", len(delete_index))

Delete index size: 280032


### Candidate Generation Using the Delete Index

For a misspelled word:

1. Generate its one-character deletions.
2. Look up each deletion in the precomputed delete index.
3. Collect the vocabulary words associated with those deletions.

The result is the Method B candidate set.

In [15]:
def method_b_candidates(word):
    """
    Method B — Symmetric Delete candidate generation.

    Two lookup passes are needed to cover all edit-distance-1 relationships:

    Pass 1 (insertion distance): look up `word` itself in the delete_index.
        Rationale: if vocab_word has `word` as one of its one-char deletions,
        then vocab_word = word + one inserted character (e.g., "hav" → "have").
        This key exists in delete_index because we stored all deletions of every
        vocab word during preprocessing.

    Pass 2 (deletion distance): delete each character of `word` one at a time
        and look up the resulting string in delete_index.
        Rationale: if that string is also a deletion of some vocab_word, then
        vocab_word = word with one character substituted (or word – 1 char).

    Together the two passes recover all vocab words within edit-distance 1.
    """
    word = word.lower()
    candidates = set()

    # Pass 1: vocab words that become `word` when one char is deleted
    #         (i.e., vocab_word is one insertion away from `word`)
    candidates.update(delete_index.get(word, set()))

    # Pass 2: vocab words reachable by deleting one char from `word`
    for i in range(len(word)):
        deleted = word[:i] + word[i + 1:]
        candidates.update(delete_index.get(deleted, set()))

    return candidates

In [16]:
print(method_b_candidates("hav"))

{'han', 'cav', 'hap', 'hat', 'has', 'haw', 'had', 'fha', 'hay', 'hal', 'ham', 'hev', 'avc', 'have'}


## 7. Comparing the Two Candidate Generators

Before implementing the complete corrector, we compare the candidate sets produced by Method A and Method B.

Ideally, both methods should identify useful vocabulary candidates, while Method B should require less computation because the vocabulary preprocessing has already been performed.

In [17]:
test_word = "hav"

a_candidates = method_a_candidates(test_word)
b_candidates = method_b_candidates(test_word)

print("Method A:", a_candidates)
print("Method B:", b_candidates)

Method A: {'han', 'cav', 'hap', 'have', 'haw', 'had', 'hay', 'hal', 'ham', 'hev', 'ha', 'has', 'hat'}
Method B: {'han', 'cav', 'hap', 'hat', 'has', 'haw', 'had', 'fha', 'hay', 'hal', 'ham', 'hev', 'avc', 'have'}


## 8. Non-Word Error Correction

A non-word error is a word that does not occur in the vocabulary.

Example:

```text
hav
sentnce

In [18]:
def best_unigram_candidate(candidates):
    if not candidates:
        return None

    return max(
        candidates,
        key=lambda word: word_counts[word]
    )

In [19]:
def correct_nonword_method_a(word):
    candidates = method_a_candidates(word)
    return best_unigram_candidate(candidates)


def correct_nonword_method_b(word):
    candidates = method_b_candidates(word)
    return best_unigram_candidate(candidates)

In [20]:
for word in ["hav", "sentnce", "teh"]:
    print(
        word,
        "→ Method A:",
        correct_nonword_method_a(word),
        "| Method B:",
        correct_nonword_method_b(word)
    )

hav → Method A: had | Method B: had
sentnce → Method A: sentence | Method B: sentence
teh → Method A: the | Method B: the


## 9. Real-Word Error Correction

Real-word errors are more difficult because the incorrect word already exists in the vocabulary.

For example:

```text
I would like to sea the world.

### Generate Candidate Corrections

For real-word errors, the original word is in the vocabulary.

We can still generate edit-distance-1 candidates and then use the bigram model to determine whether another candidate is more plausible in context.

In [21]:
def real_word_candidates(word, method="A"):
    if method.upper() == "A":
        return method_a_candidates(word)
    elif method.upper() == "B":
        return method_b_candidates(word)
    else:
        raise ValueError("Method must be 'A' or 'B'")

### Context-Based Correction

The following function compares the original word with candidate corrections using the previous word as context.

If the best candidate has a higher bigram probability than the original by the chosen threshold, we suggest the candidate.

The threshold is used to avoid changing a correct word merely because another word has a slightly higher probability.

In [22]:
import math


def real_word_context_score(previous_word, word, next_word):
    """
    Score a word using both the left and right bigram probabilities
    in log space.

        score(w) = log P(w | prev) + log P(next | w)

    With add-k smoothing in bigram_probability(), both probabilities
    are always > 0, so no epsilon floor is needed.
    """
    left_log_prob  = math.log(bigram_probability(previous_word, word))
    right_log_prob = math.log(bigram_probability(word, next_word))

    return left_log_prob + right_log_prob


def correct_real_word(
    previous_word,
    word,
    next_word,
    method="B",
    improvement_threshold=0.5
):
    """
    Correct a real-word spelling error using local bigram context.

    Compares the context score of the original word against all
    edit-distance-1 candidates. If a candidate scores more than
    improvement_threshold higher (in log space), it is selected
    as the correction.

    improvement_threshold=0.5 means the candidate must have at least
    e^0.5 ≈ 1.65x higher probability in context to be accepted.
    This prevents over-correction of valid low-frequency words.
    """
    candidates = real_word_candidates(word, method)

    if not candidates:
        return word

    original_score = real_word_context_score(
        previous_word, word, next_word
    )

    best_word  = word
    best_score = original_score

    for candidate in candidates:
        candidate_score = real_word_context_score(
            previous_word, candidate, next_word
        )

        if candidate_score > best_score:
            best_word  = candidate
            best_score = candidate_score

    improvement = best_score - original_score

    if improvement >= improvement_threshold:
        return best_word

    return word

## 10. Complete Sentence Correction

We now combine the non-word and real-word logic.

For each word:

- If the word is not in the vocabulary → treat it as a non-word error.
- If the word is in the vocabulary → it may be a real-word error, so use context.
- Otherwise, keep the original word.

We also record whether a word was changed so that the CLI can later highlight corrections.

In [23]:
def correct_sentence(sentence, method="B"):
    words = sentence.lower().split()

    corrected_words = []
    changes = []

    for i, word in enumerate(words):

        previous_word = words[i - 1] if i > 0 else "<s>"
        next_word = words[i + 1] if i < len(words) - 1 else "</s>"

        if word not in vocabulary:

            if method.upper() == "A":
                correction = correct_nonword_method_a(word)
            else:
                correction = correct_nonword_method_b(word)

            if correction is None:
                correction = word

        else:

            correction = correct_real_word(
                previous_word,
                word,
                next_word,
                method=method
            )

        corrected_words.append(correction)

        if correction != word:
            changes.append((word, correction))

    return corrected_words, changes

## 11. Test the Spelling Corrector

The assignment provides example sentences for both non-word and real-word errors.

We will test these examples before moving to large-scale evaluation.

In [24]:
examples = [
    "I hav a good feeling about this.",
    "This is a test sentnce.",
    "I would like to sea the world.",
    "Please meat me at the station."
]

for sentence in examples:
    corrected, changes = correct_sentence(sentence)

    print("Original :", sentence)
    print("Corrected:", " ".join(corrected))
    print("Changes  :", changes)
    print()

Original : I hav a good feeling about this.
Corrected: i had a good feeling about this.
Changes  : [('hav', 'had')]

Original : This is a test sentnce.
Corrected: this is a test sentence
Changes  : [('sentnce.', 'sentence')]

Original : I would like to sea the world.
Corrected: it would like to see che worlds
Changes  : [('i', 'it'), ('sea', 'see'), ('the', 'che'), ('world.', 'worlds')]

Original : Please meat me at the station.
Corrected: please beat me to the stations
Changes  : [('meat', 'beat'), ('at', 'to'), ('station.', 'stations')]



## 12. Evaluation

The spelling corrector is evaluated separately on two types of spelling errors:

1. **Non-word errors**: the corrupted word does not occur in the vocabulary.
2. **Real-word errors**: the corrupted word is itself a valid vocabulary word, but it is incorrect in context.

The evaluation uses **10% of the Brown Corpus sentences** as required by the assignment.

For each selected sentence, one alphabetic word is randomly selected and modified using a single-edit operation.

Two separate test sets are constructed so that the corrupted word satisfies the required condition for each error type.

## 12.1 Constructing the 10% Brown Corpus Test Set

The assignment specifies that **10% of the Brown Corpus sentences** should be used for evaluation.

A fixed random seed is used so that the same test sentences can be reproduced.

For each selected sentence, we attempt to create:

- one **non-word error**, where the corrupted form is not in the vocabulary;
- one **real-word error**, where the corrupted form is in the vocabulary but differs from the original word.

The sentence is searched across all eligible alphabetic words before it is considered unsuccessful. This avoids losing a sentence simply because the first randomly selected word cannot produce the required type of error.

In [25]:
import random
import time
import pandas as pd

RANDOM_SEED = 42
rng = random.Random(RANDOM_SEED)

In [26]:
total_sentences = len(processed_sentences)

num_test_sentences = int(0.10 * total_sentences)

test_sentences = rng.sample(
    processed_sentences,
    num_test_sentences
)

print("Total processed sentences :", total_sentences)
print("Required 10% test set     :", num_test_sentences)
print("Selected test sentences   :", len(test_sentences))

assert len(test_sentences) == num_test_sentences

Total processed sentences : 56766
Required 10% test set     : 5676
Selected test sentences   : 5676


### Creating Single-Edit Errors

A spelling error is introduced by applying exactly one edit operation to an original word.

The available operations are:

- deletion
- transposition
- replacement
- insertion

For the non-word test set, the corrupted word must **not** occur in the vocabulary.

For the real-word test set, the corrupted word must **occur in the vocabulary** while still being different from the original word.

In [27]:
def generate_single_edit(word, rng=None):
    """
    Generate one random candidate at exactly one edit operation away
    from the original word.

    Args:
        word: The word to corrupt.
        rng:  An optional seeded random.Random instance.
              If None, falls back to the standard `random` module
              (non-reproducible). Pass the global `rng` object when
              building reproducible test sets.
    """
    import random as _random_mod
    _rng = rng if rng is not None else _random_mod

    word = word.lower()

    operations = []

    # Deletion
    if len(word) > 1:
        for i in range(len(word)):
            operations.append(word[:i] + word[i + 1:])

    # Transposition
    for i in range(len(word) - 1):
        if word[i] != word[i + 1]:
            operations.append(
                word[:i]
                + word[i + 1]
                + word[i]
                + word[i + 2:]
            )

    # Replacement
    alphabet = "abcdefghijklmnopqrstuvwxyz"

    for i in range(len(word)):
        for char in alphabet:
            if char != word[i]:
                operations.append(
                    word[:i] + char + word[i + 1:]
                )

    # Insertion
    for i in range(len(word) + 1):
        for char in alphabet:
            operations.append(
                word[:i] + char + word[i:]
            )

    operations = list(set(operations))

    if not operations:
        return None

    return _rng.choice(operations)

In [28]:
def create_nonword_error(word, vocabulary, max_attempts=100, rng=None):
    """
    Create a single-edit corruption that is NOT in vocabulary.

    Args:
        rng: Seeded random.Random instance for reproducibility.
    """

    for _ in range(max_attempts):

        corrupted = generate_single_edit(word, rng=rng)

        if corrupted is None:
            return None

        if corrupted not in vocabulary:
            return corrupted

    return None

In [29]:
def find_nonword_error_for_sentence(sentence, vocabulary, rng):
    """
    Find a word in the sentence that can be changed by exactly
    one edit operation into a form that is NOT in the vocabulary.

    Returns an example dictionary or None.
    """

    eligible_indices = [
        i
        for i, word in enumerate(sentence)
        if word.isalpha() and len(word) >= 2
    ]

    shuffled_indices = eligible_indices.copy()
    rng.shuffle(shuffled_indices)

    for index in shuffled_indices:

        original_word = sentence[index]

        corrupted_word = create_nonword_error(
            original_word,
            vocabulary,
            rng=rng  # pass seeded rng for reproducibility
        )

        if corrupted_word is not None:

            return {
                "sentence": sentence.copy(),
                "index": index,
                "original": original_word,
                "misspelled": corrupted_word
            }

    return None

In [30]:
def _fast_edit1_vocab_candidates(word, vocabulary):
    """
    Generate all vocabulary words within edit-distance 1 of `word`
    using fast Norvig-style string enumeration (no DP table needed).

    Used ONLY for test-set corruption generation — not for the corrector.
    The corrector uses method_a_candidates (DL DP) or method_b_candidates.

    Complexity: O(L * |Σ|) string allocations, where L = word length,
    |Σ| = 26. Approximately 400 strings for an average 7-char word.
    """
    alphabet = "abcdefghijklmnopqrstuvwxyz"
    w = word.lower()
    n = len(w)
    candidates = set()

    # Deletion
    for i in range(n):
        candidates.add(w[:i] + w[i+1:])

    # Transposition
    for i in range(n - 1):
        candidates.add(w[:i] + w[i+1] + w[i] + w[i+2:])

    # Replacement
    for i in range(n):
        for c in alphabet:
            if c != w[i]:
                candidates.add(w[:i] + c + w[i+1:])

    # Insertion
    for i in range(n + 1):
        for c in alphabet:
            candidates.add(w[:i] + c + w[i:])

    return {c for c in candidates if c in vocabulary and c != w}


def create_realword_error(word, vocabulary, max_attempts=100, rng=None):
    """
    Create a single-edit corruption that IS in vocabulary
    and differs from the original word.

    Uses fast Norvig-style enumeration (not DL DP) because this function
    is called thousands of times during test-set construction. DL DP
    scanning all 40k vocab words per call would cause minutes of hang.

    Args:
        rng: Seeded random.Random instance for reproducibility.
    """
    import random as _random_mod
    _rng = rng if rng is not None else _random_mod

    valid_candidates = list(
        _fast_edit1_vocab_candidates(word, vocabulary)
    )

    if not valid_candidates:
        return None

    return _rng.choice(valid_candidates)

In [31]:
def find_realword_error_for_sentence(sentence, vocabulary, rng):
    """
    Find a word in the sentence that can be changed by exactly
    one edit operation into another vocabulary word.

    Returns an example dictionary or None.
    """

    eligible_indices = [
        i
        for i, word in enumerate(sentence)
        if word.isalpha() and len(word) >= 2
    ]

    shuffled_indices = eligible_indices.copy()
    rng.shuffle(shuffled_indices)

    for index in shuffled_indices:

        original_word = sentence[index]

        corrupted_word = create_realword_error(
            original_word,
            vocabulary,
            rng=rng  # pass seeded rng for reproducibility
        )

        if corrupted_word is not None:

            return {
                "sentence": sentence.copy(),
                "index": index,
                "original": original_word,
                "misspelled": corrupted_word
            }

    return None

In [32]:
original = "see"

corrupted = create_realword_error(
    original,
    vocabulary
)

print("Original :", original)
print("Corrupted:", corrupted)
print("In vocabulary?", corrupted in vocabulary)

Original : see
Corrupted: gee
In vocabulary? True


### Building the Non-Word Test Set

For each selected Brown sentence:

1. Select an alphabetic word.
2. Generate a single-edit corruption.
3. Keep the example only if the corrupted word is not in the vocabulary.
4. Store the original word as the gold answer.
5. Store the corrupted word as the misspelled input.
6. Store the sentence context for later evaluation.

In [33]:
nonword_examples = []
realword_examples = []

nonword_failed_sentences = []
realword_failed_sentences = []

for sentence in test_sentences:

    # Non-word version
    nonword_example = find_nonword_error_for_sentence(
        sentence,
        vocabulary,
        rng
    )

    if nonword_example is not None:
        nonword_examples.append(nonword_example)
    else:
        nonword_failed_sentences.append(sentence)

    # Real-word version
    realword_example = find_realword_error_for_sentence(
        sentence,
        vocabulary,
        rng
    )

    if realword_example is not None:
        realword_examples.append(realword_example)
    else:
        realword_failed_sentences.append(sentence)


print("Selected sentences:", len(test_sentences))

print()
print("Non-word examples:", len(nonword_examples))
print("Non-word failures:", len(nonword_failed_sentences))

print()
print("Real-word examples:", len(realword_examples))
print("Real-word failures:", len(realword_failed_sentences))

Selected sentences: 5676

Non-word examples: 5668
Non-word failures: 8

Real-word examples: 5640
Real-word failures: 36


In [34]:
assert len(test_sentences) == num_test_sentences

assert all(
    example["misspelled"] not in vocabulary
    for example in nonword_examples
)

assert all(
    example["misspelled"] in vocabulary
    and example["misspelled"] != example["original"]
    for example in realword_examples
)

print("Exactly 10% of the processed Brown sentences were selected.")
print("All non-word corruptions are outside the vocabulary.")
print("All real-word corruptions are valid vocabulary words.")

Exactly 10% of the processed Brown sentences were selected.
All non-word corruptions are outside the vocabulary.
All real-word corruptions are valid vocabulary words.


### Test Set Coverage

Some Brown sentences may contain no word that can produce the required error type through a single edit operation.

These sentences are not silently replaced with other sentences. Instead, they are recorded separately as failed constructions.

This allows the experiment to report exactly how many of the selected 10% sentences produced valid examples.

In [35]:
nonword_coverage = (
    len(nonword_examples) / len(test_sentences)
)

realword_coverage = (
    len(realword_examples) / len(test_sentences)
)

print(
    f"Non-word coverage: "
    f"{nonword_coverage:.2%}"
)

print(
    f"Real-word coverage: "
    f"{realword_coverage:.2%}"
)

Non-word coverage: 99.86%
Real-word coverage: 99.37%


In [36]:
print("NON-WORD EXAMPLES")
print("=" * 70)

for example in nonword_examples[:5]:

    print("Original   :", example["original"])
    print("Misspelled :", example["misspelled"])
    print("Sentence   :", " ".join(example["sentence"]))
    print("-" * 70)


print("\nREAL-WORD EXAMPLES")
print("=" * 70)

for example in realword_examples[:5]:

    print("Original   :", example["original"])
    print("Misspelled :", example["misspelled"])
    print("Sentence   :", " ".join(example["sentence"]))
    print("-" * 70)

NON-WORD EXAMPLES
Original   : gone
Misspelled : gonhe
Sentence   : i have gone into nursing if i care about people
----------------------------------------------------------------------
Original   : call
Misspelled : cjall
Sentence   : it is extremely doubtful that the handful of albanians who call themselves communists could have done this without the direct approval of their chinese friends
----------------------------------------------------------------------
Original   : walk
Misspelled : awlk
Sentence   : a injection inflamed a nerve and johnny can barely walk
----------------------------------------------------------------------
Original   : the
Misspelled : tye
Sentence   : he opened the door and went in pulling it shut behind him
----------------------------------------------------------------------
Original   : to
Misspelled : toc
Sentence   : the slaves were to remain as wage laborers for his account
----------------------------------------------------------------------

REA

In [37]:
print("TEST SET SUMMARY")
print("=" * 50)

print(f"Selected Brown sentences : {len(test_sentences):,}")
print(f"Non-word examples        : {len(nonword_examples):,}")
print(f"Non-word failures        : {len(nonword_failed_sentences):,}")
print(f"Non-word coverage        : {nonword_coverage:.2%}")

print()

print(f"Real-word examples       : {len(realword_examples):,}")
print(f"Real-word failures       : {len(realword_failed_sentences):,}")
print(f"Real-word coverage       : {realword_coverage:.2%}")

TEST SET SUMMARY
Selected Brown sentences : 5,676
Non-word examples        : 5,668
Non-word failures        : 8
Non-word coverage        : 99.86%

Real-word examples       : 5,640
Real-word failures       : 36
Real-word coverage       : 99.37%


## 11. Accuracy Evaluation

Each test example has a known gold-standard word: the original word before corruption.

A prediction is counted as correct when:

\[
\text{predicted word} = \text{original word}
\]

Accuracy is calculated as:

\[
Accuracy =
\frac{\text{Number of correctly corrected words}}
{\text{Total number of test words}}
\]

The two candidate-generation methods are evaluated independently.

In [38]:
def evaluate_nonword_examples(examples, method):
    """
    Evaluate a spelling correction method on non-word errors.
    """

    correct = 0
    predictions = []

    for example in examples:

        misspelled = example["misspelled"]
        gold = example["original"]

        if method.upper() == "A":
            prediction = correct_nonword_method_a(misspelled)
        else:
            prediction = correct_nonword_method_b(misspelled)

        if prediction is None:
            prediction = misspelled

        predictions.append(prediction)

        if prediction == gold:
            correct += 1

    accuracy = correct / len(examples) if examples else 0

    return accuracy, predictions

In [39]:
nonword_accuracy_a, nonword_predictions_a = evaluate_nonword_examples(
    nonword_examples,
    "A"
)

nonword_accuracy_b, nonword_predictions_b = evaluate_nonword_examples(
    nonword_examples,
    "B"
)

print(f"Method A non-word accuracy: {nonword_accuracy_a:.2%}")
print(f"Method B non-word accuracy: {nonword_accuracy_b:.2%}")

Method A non-word accuracy: 89.03%
Method B non-word accuracy: 34.19%


In [40]:
def evaluate_realword_examples(examples, method):
    """
    Evaluate a spelling correction method on real-word errors.
    """

    correct = 0
    predictions = []

    for example in examples:

        sentence = example["sentence"]
        index = example["index"]

        misspelled = example["misspelled"]
        gold = example["original"]

        previous_word = (
            sentence[index - 1]
            if index > 0
            else "<s>"
        )

        next_word = (
            sentence[index + 1]
            if index < len(sentence) - 1
            else "</s>"
        )

        prediction = correct_real_word(
            previous_word,
            misspelled,
            next_word,
            method=method
        )

        predictions.append(prediction)

        if prediction == gold:
            correct += 1

    accuracy = correct / len(examples) if examples else 0

    return accuracy, predictions

In [41]:
realword_accuracy_a, realword_predictions_a = evaluate_realword_examples(
    realword_examples,
    "A"
)

realword_accuracy_b, realword_predictions_b = evaluate_realword_examples(
    realword_examples,
    "B"
)

print(f"Method A real-word accuracy: {realword_accuracy_a:.2%}")
print(f"Method B real-word accuracy: {realword_accuracy_b:.2%}")

Method A real-word accuracy: 93.40%
Method B real-word accuracy: 63.88%


In [42]:
accuracy_results = pd.DataFrame({
    "Method": ["Method A", "Method B"],
    "Non-word Accuracy": [
        nonword_accuracy_a,
        nonword_accuracy_b
    ],
    "Real-word Accuracy": [
        realword_accuracy_a,
        realword_accuracy_b
    ]
})

accuracy_results

,Method,Non-word Accuracy,Real-word Accuracy
0,Method A,0.890261,0.934043
1,Method B,0.341920,0.638830


In [43]:
def show_errors(examples, predictions, max_examples=10):

    errors_shown = 0

    for example, prediction in zip(examples, predictions):

        if prediction != example["original"]:

            print("Original word :", example["original"])
            print("Misspelled    :", example["misspelled"])
            print("Prediction    :", prediction)
            print("Sentence      :", " ".join(example["sentence"]))
            print("-" * 60)

            errors_shown += 1

            if errors_shown >= max_examples:
                break

In [44]:
print("METHOD A — NON-WORD ERRORS")
show_errors(
    nonword_examples,
    nonword_predictions_a
)

METHOD A — NON-WORD ERRORS
Original word : this
Misspelled    : ghis
Prediction    : his
Sentence      : this difficulty arises even though we can give examples of men who have actually followed this course
------------------------------------------------------------
Original word : any
Misspelled    : aty
Prediction    : at
Sentence      : against this invincible determination to communize the whole world stands a group of nations unable to agree on fundamentals and each refusing to make any sacrifice of sovereignty for the common good of all
------------------------------------------------------------
Original word : many
Misspelled    : bany
Prediction    : any
Sentence      : to many of us this is a land to which we or our parents fled from totalitarian terror in order to live in dignified freedom
------------------------------------------------------------
Original word : self
Misspelled    : selq
Prediction    : sell
Sentence      : his old self
----------------------------------

In [45]:
print("METHOD B — NON-WORD ERRORS")
show_errors(
    nonword_examples,
    nonword_predictions_b
)

METHOD B — NON-WORD ERRORS
Original word : gone
Misspelled    : gonhe
Prediction    : agone
Sentence      : i have gone into nursing if i care about people
------------------------------------------------------------
Original word : call
Misspelled    : cjall
Prediction    : calls
Sentence      : it is extremely doubtful that the handful of albanians who call themselves communists could have done this without the direct approval of their chinese friends
------------------------------------------------------------
Original word : walk
Misspelled    : awlk
Prediction    : talk
Sentence      : a injection inflamed a nerve and johnny can barely walk
------------------------------------------------------------
Original word : to
Misspelled    : toc
Prediction    : two
Sentence      : the slaves were to remain as wage laborers for his account
------------------------------------------------------------
Original word : any
Misspelled    : aty
Prediction    : may
Sentence      : against this i

## 12. Speed Demon Benchmark

The Speed Demon experiment compares the computational efficiency of the two candidate-generation methods.

Exactly **1,000 misspelled words** are placed in a single benchmark batch.

The **same batch** is passed to:

- Method A: standard edit-distance-1 generation
- Method B: symmetric-delete candidate lookup

The total execution time of each method is measured using `time.perf_counter()`.

The speedup is calculated as:

\[
Speedup =
\frac{Time_{Method\ A}}
{Time_{Method\ B}}
\]

A speedup greater than 1 means that Method B is faster than Method A.

In [46]:
benchmark_size = 1000

# Combine the misspelled words we already generated
available_misspelled_words = (
    [example["misspelled"] for example in nonword_examples]
    + [example["misspelled"] for example in realword_examples]
)

print("Available misspelled words:", len(available_misspelled_words))

Available misspelled words: 11308


In [47]:
def create_benchmark_words(vocabulary, size=1000, seed=42):
    """
    Generate exactly `size` misspelled words.

    Each word is created using a single edit operation
    from a vocabulary word.
    """

    rng = random.Random(seed)

    vocabulary_list = [
        word for word in vocabulary
        if word.isalpha() and len(word) >= 2
    ]

    benchmark_words = []

    while len(benchmark_words) < size:

        original = rng.choice(vocabulary_list)

        corrupted = generate_single_edit(original)

        if corrupted is None:
            continue

        # Keep only misspellings that are not valid vocabulary words.
        if corrupted not in vocabulary:
            benchmark_words.append(corrupted)

    return benchmark_words

In [48]:
benchmark_words = create_benchmark_words(
    vocabulary,
    size=1000,
    seed=42
)

print("Benchmark size:", len(benchmark_words))
print("First 10 benchmark words:")
print(benchmark_words[:10])

Benchmark size: 1000
First 10 benchmark words:
['parxanormal', 'printa', 'bariuzm', 'adenda', 'whitvns', 'argonmuts', 'itiuerary', 'tensizon', 'ordinancesz', 'bubbllng']


In [49]:
assert len(benchmark_words) == 1000
print("Exactly 1,000 misspelled words prepared.")

Exactly 1,000 misspelled words prepared.


### Fair Comparison

To ensure a fair comparison, both methods receive exactly the same list of 1,000 misspelled words.

No words are added, removed, or reordered between the two experiments.

Only the candidate-generation method changes.

In [50]:
start_time_a = time.perf_counter()

method_a_results = [
    correct_nonword_method_a(word)
    for word in benchmark_words
]

end_time_a = time.perf_counter()

method_a_time = end_time_a - start_time_a

print(f"Method A total time: {method_a_time:.6f} seconds")

Method A total time: 0.254682 seconds


In [51]:
start_time_b = time.perf_counter()

method_b_results = [
    correct_nonword_method_b(word)
    for word in benchmark_words
]

end_time_b = time.perf_counter()

method_b_time = end_time_b - start_time_b

print(f"Method B total time: {method_b_time:.6f} seconds")

Method B total time: 0.010269 seconds


In [52]:
assert len(method_a_results) == 1000
assert len(method_b_results) == 1000

print("Method A processed:", len(method_a_results))
print("Method B processed:", len(method_b_results))
print("Same benchmark batch size used for both methods.")

Method A processed: 1000
Method B processed: 1000
Same benchmark batch size used for both methods.


In [53]:
speedup = (
    method_a_time / method_b_time
    if method_b_time > 0
    else float("inf")
)

print(f"Method A: {method_a_time:.6f} seconds")
print(f"Method B: {method_b_time:.6f} seconds")
print(f"Speedup: {speedup:.2f}x")

Method A: 0.254682 seconds
Method B: 0.010269 seconds
Speedup: 24.80x


In [54]:
speed_results = pd.DataFrame({
    "Method": [
        "Method A - Edit Distance 1",
        "Method B - Symmetric Delete"
    ],
    "Test Words": [
        1000,
        1000
    ],
    "Total Time (seconds)": [
        method_a_time,
        method_b_time
    ],
    "Average Time (ms/word)": [
        (method_a_time / 1000) * 1000,
        (method_b_time / 1000) * 1000
    ]
})

speed_results

,Method,Test Words,Total Time (seconds),Average Time (ms/word)
0,Method A - Edit Distance 1,1000,0.254682,0.254682
1,Method B - Symmetric Delete,1000,0.010269,0.010269


In [55]:
# Candidate set size comparison
method_a_candidate_counts = [
    len(method_a_candidates(word))
    for word in benchmark_words
]

method_b_candidate_counts = [
    len(method_b_candidates(word))
    for word in benchmark_words
]

average_candidates_a = (
    sum(method_a_candidate_counts)
    / len(method_a_candidate_counts)
)

average_candidates_b = (
    sum(method_b_candidate_counts)
    / len(method_b_candidate_counts)
)

print(
    f"Average candidates per word - Method A: "
    f"{average_candidates_a:.2f}"
)

print(
    f"Average candidates per word - Method B: "
    f"{average_candidates_b:.2f}"
)

Average candidates per word - Method A: 1.38
Average candidates per word - Method B: 1.39


### Speed Difference Analysis

Method A generates possible spelling corrections dynamically for every query by constructing deletion, transposition, replacement, and insertion variants and then checking them against the vocabulary.

Method B performs an expensive preprocessing step once. During preprocessing, every one-character deletion of every vocabulary word is stored in a symmetric-delete index.

At query time, Method B only needs to:

1. Generate the one-character deletions of the misspelled word.
2. Look up those deletions in the precomputed index.
3. Retrieve the associated vocabulary words.

Therefore, Method B moves computational work from query time to preprocessing time. This makes it particularly efficient when many misspelled words must be processed against the same vocabulary.

The benchmark measures this difference using the same 1,000-word input batch for both methods.

## 14. Interactive Terminal CLI

The assignment requires a continuous terminal application.

The application must:

- Ask the user for a sentence.
- Correct the sentence.
- Highlight changed words.
- Display correction latency.
- Stop when the user enters `exit`.

We first implement the display logic in the notebook. The same function can later be moved into `cli.py`.

In [56]:
def format_correction(original_words, corrected_words):
    """
    Format the corrected sentence, wrapping changed words in **asterisks**
    as required by the assignment (Part 5 highlighting requirement).
    """
    output = []

    for original, corrected in zip(original_words, corrected_words):
        if original != corrected:
            output.append(f"**{corrected}**")
        else:
            output.append(corrected)

    return " ".join(output)


def interactive_correction():
    """
    Continuous Terminal CLI — required by the assignment (Part 5).

    Runs a while-loop, asking for sentences and correcting them.
    Changed words are wrapped in **ASTERISKS**.
    Stops when the user types 'exit'.

    Note: Cannot be run interactively inside a Jupyter notebook cell
    (stdin is not available). Copy this function into cli.py and run:
        python cli.py
    """
    print("NLP Spelling Corrector")
    print("Type 'exit' to quit.")

    while True:
        sentence = input("\nEnter sentence: ")

        if sentence.strip().lower() == "exit":
            print("Exiting...")
            break

        original_words = sentence.lower().split()

        start = time.perf_counter()

        corrected_words, changes = correct_sentence(sentence)

        latency = (time.perf_counter() - start) * 1000

        highlighted = format_correction(
            original_words,
            corrected_words
        )

        print("Corrected:", highlighted)
        print(f"Latency: {latency:.3f} ms")

## 15. Final Sample Runs

The following examples demonstrate the final behavior required by the assignment.

We test:

- Non-word errors
- Real-word errors
- Highlighted corrections
- Latency reporting

In [57]:
test_sentences = [
    "I hav a good feeling about this.",
    "This is a test sentnce.",
    "I would like to sea the world.",
    "Please meat me at the station."
]

for sentence in test_sentences:
    original_words = sentence.lower().split()

    start = time.perf_counter()
    corrected_words, changes = correct_sentence(sentence)
    latency = (time.perf_counter() - start) * 1000

    print("Input    :", sentence)
    print("Corrected:", format_correction(original_words, corrected_words))
    print(f"Latency  : {latency:.3f} ms")
    print()

Input    : I hav a good feeling about this.
Corrected: i **had** a good feeling about this.
Latency  : 0.526 ms

Input    : This is a test sentnce.
Corrected: this is a test **sentence**
Latency  : 0.539 ms

Input    : I would like to sea the world.
Corrected: **it** would like to **see** **che** **worlds**
Latency  : 0.915 ms

Input    : Please meat me at the station.
Corrected: please **beat** me **to** the **stations**
Latency  : 0.572 ms



# 16. Conclusion

The spelling corrector implements the required components:

1. Brown Corpus vocabulary and unigram frequency model
2. Bigram language model
3. Standard Edit Distance 1 candidate generation
4. Symmetric Delete candidate generation
5. Non-word error correction
6. Real-word error correction using contextual probabilities
7. Accuracy evaluation
8. 1,000-word Speed Demon benchmark
9. Interactive correction interface

## Method Comparison

Method A generates candidates directly using the four edit operations.

Method B uses a precomputed deletion index, allowing candidate lookup to be performed more efficiently after preprocessing.

The benchmark results are used to determine the actual runtime difference between the two approaches.

## Final Observation

The final choice of candidate-generation method should consider both correction quality and latency. Method B is expected to be particularly useful when the vocabulary is large and many words must be corrected repeatedly.